# NLP Project 1 – LSTM Text Generation on Shakespeare
**Group:** Kevin Bajul, Asma El Moushine, Sébastien Le Masne de Chermont, Marin Mathé

---
## Roadmap
1. Data loading & train/val/test split
2. Vocabulary & encoding
3. Dataset & DataLoader
4. Model definitions  (RNN baseline · 1-layer LSTM · 2-layer LSTM · GRU)
5. Training loop  (with perplexity tracking)
6. Evaluation  (loss, perplexity, BLEU, % correctly-spelled words, n-gram overlap)
7. Text generation  (temperature scaling + Nucleus / Top-p sampling)
8. Hyperparameter experiments  (hidden size, batch size, learning rate, regularisation)
9. Results & qualitative comparison
---

## 0 · Imports & reproducibility

In [1]:
# Install PyTorch and related packages into the current kernel
%pip install torch torchvision torchaudio pyspellchecker nltk --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, math, random, time, re
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

import nltk
nltk.download('punkt', quiet=True)
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from spellchecker import SpellChecker

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

OSError: [WinError 1114] Une routine d’initialisation d’une bibliothèque de liens dynamiques (DLL) a échoué. Error loading "c:\Users\asmae\Documents\anaconda\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

---
## 1 · Data loading & train / val / test split

We use **Shakespeare's plays** (`shakespeare.txt`). The file is split  
**80 % train · 10 % validation · 10 % test** at the *character* level.

In [ ]:
# ── 1.1  Load raw text ────────────────────────────────────────────────────────
def load_text(path: str) -> str:
    with open(path, 'r', encoding='utf-8') as f:
        return f.read()


# ── 1.2  Character-level split ────────────────────────────────────────────────
def split_text(text: str, train_ratio=0.8, val_ratio=0.1):
    """
    Returns (train_text, val_text, test_text) as plain strings.
    Splits are contiguous (no shuffling) to preserve sequential structure.
    """
    n = len(text)
    train_end = int(n * train_ratio)
    val_end   = int(n * (train_ratio + val_ratio))
    return text[:train_end], text[train_end:val_end], text[val_end:]


# ── Run ───────────────────────────────────────────────────────────────────────
TEXT_PATH = 'shakespeare.txt'        
text      = load_text(TEXT_PATH)
train_text, val_text, test_text = split_text(text)

print(f'Total chars : {len(text):,}')
print(f'Train       : {len(train_text):,}')
print(f'Validation  : {len(val_text):,}')
print(f'Test        : {len(test_text):,}')

---
## 2 · Vocabulary & character encoding

We build a **character-level** vocabulary from the *training set only*,
then reuse those mappings for val and test.

In [ ]:
# ── 2.1  Build vocabulary ─────────────────────────────────────────────────────
def build_vocab(text: str):
    """
    Returns (vocab_size, char_to_idx, idx_to_char) built from `text`.
    Characters are sorted for determinism.
    """
    chars       = sorted(set(text))
    char_to_idx = {ch: i for i, ch in enumerate(chars)}
    idx_to_char = {i: ch for i, ch in enumerate(chars)}
    return len(chars), char_to_idx, idx_to_char


# ── 2.2  Encode / decode helpers ──────────────────────────────────────────────
def encode(text: str, char_to_idx: dict) -> list:
    """Convert string → list of integer indices."""
    return [char_to_idx[ch] for ch in text if ch in char_to_idx]


def decode(indices, idx_to_char: dict) -> str:
    """Convert list / tensor of indices → string."""
    return ''.join(idx_to_char[int(i)] for i in indices)


# ── Run ───────────────────────────────────────────────────────────────────────
vocab_size, char_to_idx, idx_to_char = build_vocab(train_text)

train_ids = encode(train_text, char_to_idx)
val_ids   = encode(val_text,   char_to_idx)
test_ids  = encode(test_text,  char_to_idx)

print(f'Vocabulary size : {vocab_size}')
print(f'Sample chars    : {list(idx_to_char.values())[:20]}')

---
## 3 · Dataset & DataLoader

We create **non-overlapping chunks** of `seq_length` characters.  
Each sample is `(input_chunk, target_chunk)` where target is shifted by 1.

In [ ]:
class CharDataset(Dataset):
    """
    Returns (x, y) pairs of integer tensors, each of length `seq_length`.
    y is x shifted right by one position (next-character prediction).
    """
    def __init__(self, ids: list, seq_length: int):
        self.data       = torch.tensor(ids, dtype=torch.long)
        self.seq_length = seq_length

    def __len__(self):
        return (len(self.data) - 1) // self.seq_length

    def __getitem__(self, idx):
        start = idx * self.seq_length
        x = self.data[start       : start + self.seq_length]
        y = self.data[start + 1   : start + self.seq_length + 1]
        return x, y


def make_loaders(train_ids, val_ids, test_ids,
                 seq_length=100, batch_size=64, num_workers=0):
    train_ds = CharDataset(train_ids, seq_length)
    val_ds   = CharDataset(val_ids,   seq_length)
    test_ds  = CharDataset(test_ids,  seq_length)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=num_workers, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=num_workers, drop_last=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=num_workers, drop_last=True)
    return train_loader, val_loader, test_loader


# ── Default hyper-parameters ──────────────────────────────────────────────────
SEQ_LENGTH = 100
BATCH_SIZE = 64

train_loader, val_loader, test_loader = make_loaders(
    train_ids, val_ids, test_ids, SEQ_LENGTH, BATCH_SIZE
)
print(f'Train batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')
print(f'Test  batches : {len(test_loader)}')

---
## 4 · Model definitions

All four models share **the same interface** so we can swap them without changing the training loop:
- `forward(x, hidden)` → `(logits, new_hidden)`
- `init_hidden(batch_size)` → zero hidden state

In [ ]:
# ── 4.1  Vanilla RNN (baseline, mirrors Assignment 4 logic) ───────────────────
class VanillaRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_layers=1, dropout=0.0):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers
        self.embed  = nn.Embedding(vocab_size, vocab_size)   # one-hot-like embedding
        self.rnn    = nn.RNN(vocab_size, hidden_size,
                             num_layers=num_layers,
                             batch_first=True,
                             dropout=dropout if num_layers > 1 else 0.0)
        self.fc     = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden):
        emb    = self.embed(x)                  # (B, T, vocab_size)
        out, h = self.rnn(emb, hidden)          # out: (B, T, H)
        logits = self.fc(out)                   # (B, T, vocab_size)
        return logits, h

    def init_hidden(self, batch_size):
        return torch.zeros(self.num_layers, batch_size, self.hidden_size, device=DEVICE)


# ── 4.2  LSTM (1 or 2 layers, optional dropout) ───────────────────────────────
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_layers=1, dropout=0.0):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers
        self.embed  = nn.Embedding(vocab_size, vocab_size)
        self.lstm   = nn.LSTM(vocab_size, hidden_size,
                              num_layers=num_layers,
                              batch_first=True,
                              dropout=dropout if num_layers > 1 else 0.0)
        self.fc     = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden):
        emb    = self.embed(x)
        out, h = self.lstm(emb, hidden)         # h = (h_n, c_n)
        logits = self.fc(out)
        return logits, h

    def init_hidden(self, batch_size):
        h = torch.zeros(self.num_layers, batch_size, self.hidden_size, device=DEVICE)
        c = torch.zeros(self.num_layers, batch_size, self.hidden_size, device=DEVICE)
        return (h, c)


# ── 4.3  GRU (same interface as LSTM) ─────────────────────────────────────────
class GRUModel(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_layers=1, dropout=0.0):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers
        self.embed  = nn.Embedding(vocab_size, vocab_size)
        self.gru    = nn.GRU(vocab_size, hidden_size,
                             num_layers=num_layers,
                             batch_first=True,
                             dropout=dropout if num_layers > 1 else 0.0)
        self.fc     = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden):
        emb    = self.embed(x)
        out, h = self.gru(emb, hidden)
        logits = self.fc(out)
        return logits, h

    def init_hidden(self, batch_size):
        return torch.zeros(self.num_layers, batch_size, self.hidden_size, device=DEVICE)


# ── Quick sanity-check ────────────────────────────────────────────────────────
for name, Model, layers in [
    ('RNN',      VanillaRNN, 1),
    ('LSTM-1L',  LSTMModel,  1),
    ('LSTM-2L',  LSTMModel,  2),
    ('GRU',      GRUModel,   1),
]:
    m   = Model(vocab_size, hidden_size=128, num_layers=layers).to(DEVICE)
    x   = torch.randint(0, vocab_size, (4, 10)).to(DEVICE)
    h   = m.init_hidden(4)
    out, _ = m(x, h)
    params = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'{name:8s}  output={tuple(out.shape)}  params={params:,}')

---
## 5 · Training loop

A single `train_model` function works for **any** model above.  
It returns per-epoch train & val loss (and perplexity) for later plotting.

In [ ]:
def detach_hidden(hidden):
    """Detach hidden state from computation graph (truncated BPTT)."""
    if isinstance(hidden, tuple):          # LSTM → (h, c)
        return tuple(h.detach() for h in hidden)
    return hidden.detach()                 # RNN / GRU


def run_epoch(model, loader, criterion, optimizer=None, clip=1.0):
    """
    One forward (+ optional backward) pass over `loader`.
    If optimizer is None → evaluation mode.
    Returns (avg_loss, perplexity).
    """
    training = optimizer is not None
    model.train(training)

    total_loss, n_batches = 0.0, 0

    for x_batch, y_batch in loader:
        x_batch = x_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        hidden = model.init_hidden(x_batch.size(0))

        if training:
            optimizer.zero_grad()

        logits, hidden = model(x_batch, hidden)
        hidden = detach_hidden(hidden)

        # logits : (B, T, V)  →  (B*T, V)
        # y      : (B, T)     →  (B*T,)
        loss = criterion(logits.reshape(-1, logits.size(-1)),
                         y_batch.reshape(-1))

        if training:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), clip)
            optimizer.step()

        total_loss += loss.item()
        n_batches  += 1

    avg_loss   = total_loss / n_batches
    perplexity = math.exp(avg_loss)
    return avg_loss, perplexity


def train_model(model, train_loader, val_loader,
                n_epochs=10, lr=0.001, clip=1.0,
                patience=3, verbose=True):
    """
    Full training loop with early stopping.

    Returns
    -------
    history : dict with keys 'train_loss', 'val_loss', 'train_ppl', 'val_ppl'
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {'train_loss': [], 'val_loss': [], 'train_ppl': [], 'val_ppl': []}
    best_val_loss = float('inf')
    no_improve    = 0

    for epoch in range(1, n_epochs + 1):
        t0 = time.time()
        train_loss, train_ppl = run_epoch(model, train_loader, criterion, optimizer, clip)
        val_loss,   val_ppl   = run_epoch(model, val_loader,   criterion, None,      clip)

        history['train_loss'].append(train_loss)
        history['val_loss'  ].append(val_loss)
        history['train_ppl' ].append(train_ppl)
        history['val_ppl'   ].append(val_ppl)

        if verbose:
            print(f'Epoch {epoch:3d}/{n_epochs}  '
                  f'train_loss={train_loss:.4f} (ppl={train_ppl:.1f})  '
                  f'val_loss={val_loss:.4f} (ppl={val_ppl:.1f})  '
                  f'[{time.time()-t0:.0f}s]')

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            no_improve    = 0
            torch.save(model.state_dict(), 'best_model.pt')
        else:
            no_improve += 1
            if no_improve >= patience:
                if verbose:
                    print(f'  Early stopping at epoch {epoch}.')
                break

    # Reload best weights
    model.load_state_dict(torch.load('best_model.pt', map_location=DEVICE))
    return history

---
## 6 · Evaluation metrics

Beyond loss/perplexity, we compute:
- **% correctly-spelled words** (using `pyspellchecker`)
- **n-gram overlap** with the training corpus (n = 2, 3)
- **BLEU score** against a reference chunk of the test set

In [ ]:
# ── 6.1  Loss & perplexity on any split ──────────────────────────────────────
def evaluate_loss(model, loader):
    criterion = nn.CrossEntropyLoss()
    loss, ppl = run_epoch(model, loader, criterion, optimizer=None)
    return {'loss': loss, 'perplexity': ppl}


# ── 6.2  Spelling accuracy ────────────────────────────────────────────────────
def spelling_accuracy(text: str) -> float:
    """
    Fraction of word tokens that are correctly spelled (English dictionary).
    Ignores punctuation-only tokens.
    """
    spell  = SpellChecker()
    words  = re.findall(r"[a-zA-Z']+", text.lower())
    if not words:
        return 0.0
    misspelled = spell.unknown(words)
    return 1.0 - len(misspelled) / len(words)


# ── 6.3  N-gram overlap (generated vs. reference corpus) ─────────────────────
def ngram_overlap(generated: str, reference: str, n: int) -> float:
    """
    Fraction of n-grams in `generated` that also appear in `reference`.
    Operates at the *character* level.
    """
    def get_ngrams(s, n):
        return Counter(s[i:i+n] for i in range(len(s) - n + 1))

    gen_ngrams = get_ngrams(generated, n)
    ref_ngrams = get_ngrams(reference, n)

    if not gen_ngrams:
        return 0.0
    overlap = sum(min(cnt, ref_ngrams[ng]) for ng, cnt in gen_ngrams.items())
    return overlap / sum(gen_ngrams.values())


# ── 6.4  BLEU score ──────────────────────────────────────────────────────────
def compute_bleu(generated: str, reference: str) -> float:
    """
    Character-level BLEU between generated and reference text.
    Uses smoothing to handle short sequences.
    """
    smooth  = SmoothingFunction().method1
    ref_tok = list(reference)
    hyp_tok = list(generated)
    return sentence_bleu([ref_tok], hyp_tok, smoothing_function=smooth)


# ── 6.5  All-in-one evaluation ────────────────────────────────────────────────
def full_evaluate(model, loader, generated_text: str, reference_text: str):
    """
    Returns a dict with all metrics for the report.
    `generated_text`  : output of generate_text(model, ...)
    `reference_text`  : a chunk of training / test text
    """
    metrics = evaluate_loss(model, loader)
    metrics['spelling_acc'] = spelling_accuracy(generated_text)
    metrics['bigram_overlap']  = ngram_overlap(generated_text, reference_text, n=2)
    metrics['trigram_overlap'] = ngram_overlap(generated_text, reference_text, n=3)
    metrics['bleu']            = compute_bleu(generated_text,  reference_text)
    return metrics



---
## 7 · Text generation

Two sampling strategies:
- **Temperature scaling** – divides logits by `T` before softmax
- **Nucleus sampling (Top-p)** – restricts sampling to the smallest set of tokens whose cumulative probability ≥ `p` (Holtzman et al., ICLR 2020)

In [ ]:
# ── 7.1  Generate text ────────────────────────────────────────────────────────
@torch.no_grad()
def generate_text(model, seed_text: str, char_to_idx: dict, idx_to_char: dict,
                  length=500, temperature=1.0, top_p=1.0) -> str:
    """
    Generate `length` characters starting from `seed_text`.

    Parameters
    ----------
    temperature : float
        < 1 → more deterministic; > 1 → more random.
    top_p : float
        Nucleus sampling threshold (1.0 = no nucleus filtering = pure temperature).
    """
    model.eval()

    # Encode seed
    input_ids = [char_to_idx.get(ch, 0) for ch in seed_text]
    input_tensor = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0).to(DEVICE)  # (1, T_seed)

    # Prime the hidden state with the seed
    hidden = model.init_hidden(1)
    _, hidden = model(input_tensor, hidden)

    # Start generating from last seed character
    generated = list(seed_text)
    current   = torch.tensor([[input_ids[-1]]], dtype=torch.long).to(DEVICE)  # (1, 1)

    for _ in range(length):
        logits, hidden = model(current, hidden)   # logits: (1, 1, V)
        logits = logits.squeeze() / temperature   # (V,)

        # --- Nucleus / Top-p filtering ---
        if top_p < 1.0:
            sorted_logits, sorted_idx = torch.sort(logits, descending=True)
            probs = torch.softmax(sorted_logits, dim=-1)
            cum_probs = torch.cumsum(probs, dim=0)

            # Remove tokens beyond the nucleus
            remove_mask = cum_probs > top_p
            remove_mask[1:] = remove_mask[:-1].clone()   # keep the token that pushes over threshold
            remove_mask[0]  = False
            sorted_logits[remove_mask] = float('-inf')

            # Scatter back to original ordering
            logits = torch.zeros_like(logits).scatter_(0, sorted_idx, sorted_logits)

        probs = torch.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)

        generated.append(idx_to_char[next_id.item()])
        current = next_id.unsqueeze(0)            # (1, 1)

    return ''.join(generated)




---
## 8 · Experiments

### 8.A — Train all four architectures (default hyper-parameters)

In [ ]:
# ── Shared defaults ───────────────────────────────────────────────────────────
HIDDEN_SIZE = 256
N_EPOCHS    = 20
LR          = 0.001
DROPOUT     = 0.3     # only active when num_layers > 1

configs = [
    ('RNN',     VanillaRNN, dict(num_layers=1, dropout=0.0)),
    ('LSTM-1L', LSTMModel,  dict(num_layers=1, dropout=0.0)),
    ('LSTM-2L', LSTMModel,  dict(num_layers=2, dropout=DROPOUT)),
    ('GRU',     GRUModel,   dict(num_layers=1, dropout=0.0)),
]

all_histories = {}
all_models    = {}

for name, ModelClass, kwargs in configs:
    print(f'\n{"="*55}')
    print(f'  Training {name}')
    print(f'{"="*55}')

    model = ModelClass(vocab_size, HIDDEN_SIZE, **kwargs).to(DEVICE)
    history = train_model(model, train_loader, val_loader,
                          n_epochs=N_EPOCHS, lr=LR, patience=3)
    all_histories[name] = history
    all_models[name]    = model



### 8.B — Plot training & validation loss / perplexity

In [ ]:
def plot_histories(histories, metric='val_loss', ylabel='Validation loss',
                   save_path=None):
    plt.figure(figsize=(10, 5))
    for name, h in histories.items():
        plt.plot(h[metric], label=name)
    plt.xlabel('Epoch')
    plt.ylabel(ylabel)
    plt.title(ylabel + ' per epoch')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()


plot_histories(all_histories, 'train_loss', 'Train loss',        'train_loss.png')
plot_histories(all_histories, 'val_loss',   'Validation loss',   'val_loss.png')
plot_histories(all_histories, 'val_ppl',    'Validation perplexity', 'val_ppl.png')

### 8.C — Quantitative evaluation on the test set

In [ ]:
SEED_TEXT  = 'To be, or not to be, that is the question:'   # classic Shakespeare prompt
GEN_LENGTH = 1000
REFERENCE  = test_text[:GEN_LENGTH]                          # same length for fair BLEU

results = {}
for name, model in all_models.items():
    gen_text = generate_text(model, SEED_TEXT, char_to_idx, idx_to_char, GEN_LENGTH)
    metrics  = full_evaluate(model, test_loader, gen_text, REFERENCE)
    results[name] = metrics
    print(f'{name:8s}  ppl={metrics["perplexity"]:7.2f}  '
          f'spell={metrics["spelling_acc"]*100:.1f}%  '
          f'bigram={metrics["bigram_overlap"]*100:.1f}%  '
          f'trigram={metrics["trigram_overlap"]*100:.1f}%  '
          f'BLEU={metrics["bleu"]:.4f}')

### 8.D — Qualitative: generated text samples

In [ ]:
for name, model in all_models.items():
    print(f'\n{"─"*60}')
    print(f' {name}  (temperature=1.0, top_p=1.0)')
    print(f'{"─"*60}')
    print(generate_text(model, SEED_TEXT, char_to_idx, idx_to_char,
                        length=300, temperature=1.0, top_p=1.0))

### 8.E — Temperature & Nucleus sampling investigation (best model)

In [ ]:
# Pick the best model by test perplexity
best_name  = min(results, key=lambda k: results[k]['perplexity'])
best_model = all_models[best_name]
print(f'Best model : {best_name}')

# ── Temperature sweep ─────────────────────────────────────────────────────────
for temp in [0.5, 0.8, 1.0, 1.2, 1.5]:
    print(f'\n── temperature={temp} ────────────────────')
    print(generate_text(best_model, SEED_TEXT, char_to_idx, idx_to_char,
                        length=200, temperature=temp, top_p=1.0))

# ── Nucleus sampling sweep ────────────────────────────────────────────────────
for top_p in [0.7, 0.9, 0.95]:
    print(f'\n── nucleus top_p={top_p} (temperature=1.0) ──────')
    print(generate_text(best_model, SEED_TEXT, char_to_idx, idx_to_char,
                        length=200, temperature=1.0, top_p=top_p))

### 8.F — Hidden size investigation

In [ ]:
hidden_results = {}

for h_size in [64, 128, 256, 512]:
    print(f'\n── hidden_size={h_size} ────────────────────')
    m = LSTMModel(vocab_size, h_size, num_layers=1).to(DEVICE)
    hist = train_model(m, train_loader, val_loader,
                       n_epochs=N_EPOCHS, lr=LR, patience=3, verbose=False)
    metrics = evaluate_loss(m, test_loader)
    hidden_results[h_size] = {'history': hist, 'test_ppl': metrics['perplexity']}
    print(f'  test perplexity = {metrics["perplexity"]:.2f}')

# Plot val loss curves
plt.figure(figsize=(9, 4))
for h_size, r in hidden_results.items():
    plt.plot(r['history']['val_loss'], label=f'h={h_size}')
plt.xlabel('Epoch'); plt.ylabel('Val loss')
plt.title('Effect of hidden size (1-layer LSTM)')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig('hidden_size_comparison.png', dpi=150); plt.show()

### 8.G — Hyperparameter grid search  (batch size × learning rate)

In [ ]:
from itertools import product as iproduct

hp_grid = {
    'batch_size': [32, 64, 128],
    'lr'        : [0.0005, 0.001, 0.005],
}

grid_results = []

for bs, lr in iproduct(hp_grid['batch_size'], hp_grid['lr']):
    tl, vl, tel = make_loaders(train_ids, val_ids, test_ids, SEQ_LENGTH, bs)
    m   = LSTMModel(vocab_size, 256, num_layers=1).to(DEVICE)
    hist = train_model(m, tl, vl, n_epochs=10, lr=lr, patience=2, verbose=False)
    best_val = min(hist['val_ppl'])
    test_ppl = evaluate_loss(m, tel)['perplexity']
    grid_results.append({'batch_size': bs, 'lr': lr,
                         'best_val_ppl': best_val, 'test_ppl': test_ppl})
    print(f'  bs={bs:3d}  lr={lr:.4f}  val_ppl={best_val:.1f}  test_ppl={test_ppl:.1f}')

# Summary
best_hp = min(grid_results, key=lambda r: r['test_ppl'])
print(f'\nBest HP: batch_size={best_hp["batch_size"]} lr={best_hp["lr"]}  test_ppl={best_hp["test_ppl"]:.2f}')

### 8.H — Regularisation: dropout investigation

In [ ]:
dropout_results = {}

for drop in [0.0, 0.2, 0.4, 0.5]:
    m = LSTMModel(vocab_size, 256, num_layers=2, dropout=drop).to(DEVICE)
    hist = train_model(m, train_loader, val_loader,
                       n_epochs=N_EPOCHS, lr=LR, patience=3, verbose=False)
    test_ppl = evaluate_loss(m, test_loader)['perplexity']
    dropout_results[drop] = {'history': hist, 'test_ppl': test_ppl}
    print(f'  dropout={drop}  test_ppl={test_ppl:.2f}')

plt.figure(figsize=(9, 4))
for drop, r in dropout_results.items():
    plt.plot(r['history']['val_loss'], label=f'dropout={drop}')
plt.xlabel('Epoch'); plt.ylabel('Val loss')
plt.title('Effect of dropout (2-layer LSTM)')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig('dropout_comparison.png', dpi=150); plt.show()

---
## 9 · Results summary & qualitative analysis

Print a clean table and generate long passages from each architecture.

In [ ]:
# ── 9.1  Results table ────────────────────────────────────────────────────────
print(f'{"Model":8s} | {"Test PPL":>10} | {"Spell%":>8} | {"Bigram%":>8} | {"Trigram%":>9} | {"BLEU":>8}')
print('-' * 62)
for name, m in results.items():
    print(f'{name:8s} | {m["perplexity"]:10.2f} | '
          f'{m["spelling_acc"]*100:7.1f}% | '
          f'{m["bigram_overlap"]*100:7.1f}% | '
          f'{m["trigram_overlap"]*100:8.1f}% | '
          f'{m["bleu"]:8.4f}')


# ── 9.2  Long qualitative passages ───────────────────────────────────────────
print('\n\n=== Long generated passages (500 chars) ===')
for name, model in all_models.items():
    print(f'\n──── {name} ────')
    print(generate_text(model, SEED_TEXT, char_to_idx, idx_to_char,
                        length=500, temperature=0.8, top_p=0.95))